Undersampling + Sliding Window

In [1]:
import pandas as pd
from sklearn.preprocessing import QuantileTransformer
import pandas as pd
import numpy as np
import os
import random
import torch

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)
train_df = pd.read_parquet(r"final_data\chunk-based-split-3\no_scaler_train_df_prepared.parquet", engine="pyarrow")
valid_df = pd.read_parquet(r"final_data\chunk-based-split-3\no_scaler_valid_df_prepared.parquet", engine="pyarrow")
test_df = pd.read_parquet(r"final_data\chunk-based-split-3\no_scaler_test_df_prepared.parquet", engine="pyarrow")
train_df["timestamp_num"] = pd.to_datetime(train_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
valid_df["timestamp_num"] = pd.to_datetime(valid_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
test_df["timestamp_num"] = pd.to_datetime(test_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9

train_df.sort_values(by='timestamp_num', inplace=True)
valid_df.sort_values(by='timestamp_num', inplace=True)
test_df.sort_values(by='timestamp_num', inplace=True)

cols_to_drop = ['flow_id', 'timestamp', 'timestamp_num', 'src_ip', 'dst_ip', 'protocol', 'src_port', 'dst_port']

train_df.drop(columns= cols_to_drop, inplace=True, errors='ignore')
valid_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
test_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
feat_cols = [col for col in train_df.columns if col != 'label']
scaler_full = QuantileTransformer(output_distribution="normal", random_state=42)
print("[Preprocessing] Đang fit_transform QuantileTransformer trên toàn bộ train_df...")
train_df[feat_cols] = scaler_full.fit_transform(train_df[feat_cols])
print("[Preprocessing] Đang transform valid_df và test_df...")
valid_df[feat_cols] = scaler_full.transform(valid_df[feat_cols])
test_df[feat_cols] = scaler_full.transform(test_df[feat_cols])

[Preprocessing] Đang fit_transform QuantileTransformer trên toàn bộ train_df...
[Preprocessing] Đang transform valid_df và test_df...


In [2]:
import torch
from torch.utils.data import Dataset
import numpy as np

class UndersampledSlidingWindowDataset(Dataset):
    def __init__(self, df, time_steps, max_samples_per_class=20000, step=5):
        self.X = torch.tensor(df.drop(columns=['label']).values, dtype=torch.float32)
        self.y = torch.tensor(df['label'].values, dtype=torch.long)
        self.time_steps = time_steps
        self.step = step
        
        print(f"Đang tính toán các cửa sổ hợp lệ (global step={step}) và Undersampling...")
        
        # tạo mảng index cho step = 1 (dành riêng cho các class hiếm cần bảo toàn)
        all_indices_step1 = np.arange(0, len(self.X) - self.time_steps + 1, 1)
        window_labels_step1 = self.y[all_indices_step1 + self.time_steps - 1].numpy()
        
        # tạo mảng index theo step được truyền vào (cho các class còn lại)
        all_indices_stepped = np.arange(0, len(self.X) - self.time_steps + 1, self.step)
        window_labels_stepped = self.y[all_indices_stepped + self.time_steps - 1].numpy()
        
        self.valid_indices = []
        
        # lặp qua từng class (Lấy từ step=1 để đảm bảo không sót class nào)
        classes = np.unique(window_labels_step1)
        
        for c in classes:
            # ƯU TIÊN GIỮ NGUYÊN băm cửa sổ 1-1 cho Class 2 và 6
            if c in [100]:
                c_indices = all_indices_step1[window_labels_step1 == c]
            else:
                c_indices = all_indices_stepped[window_labels_stepped == c]
                
            count = len(c_indices)
            
            # nếu class đó có nhiều mẫu hơn ngưỡng max_samples_per_class
            if count > max_samples_per_class:
                # chọn ngẫu nhiên không hoàn lại
                c_indices = np.random.choice(c_indices, max_samples_per_class, replace=False)
                print(f"Class {c}: Giảm từ {count} xuống {max_samples_per_class} cửa sổ.")
            else:
                # nếu ít hơn thì giữ nguyên
                if c in [2, 6]:
                    print(f"Class {c}: Giữ nguyên {count} cửa sổ (sử dụng step=1 để bảo toàn).")
                else:
                    print(f"Class {c}: Giữ nguyên {count} cửa sổ (step={self.step}).")
                
            self.valid_indices.extend(c_indices)
            
        # Trộn đều danh sách index để các batch đa dạng hơn
        np.random.shuffle(self.valid_indices)
        self.valid_indices = np.array(self.valid_indices) # Chuyển sang ndarray
        print(f"Tổng số cửa sổ sau khi lọc và Undersampling: {len(self.valid_indices)}")

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        # lấy các index đã được lọc và xáo trộn
        actual_idx = self.valid_indices[idx]
        
        # lấy feature và label của cửa sổ trượt tại index thực sự
        window_X = self.X[actual_idx : actual_idx + self.time_steps]
        label_y = self.y[actual_idx + self.time_steps - 1]
        
        return window_X, label_y

In [3]:
import torch
import numpy as np
from torch.utils.data import Dataset

class DynamicUndersampledSlidingWindowDataset(Dataset):
    def __init__(self, df, time_steps, max_samples_per_class=20000, 
                 step=5, resample_each_epoch=False, rare_classes=None):
        
        # Mặc định bảo vệ Class 2 và 6 nếu không truyền vào
        if rare_classes is None:
            rare_classes = [0, 2, 3, 4, 5, 8, 10]
            
        feat_cols = [col for col in df.columns if col != 'label']
        self.X = torch.as_tensor(df[feat_cols].to_numpy(dtype=np.float32))
        self.y = torch.as_tensor(df['label'].to_numpy(dtype=np.int64))
        
        self.time_steps = time_steps
        self.step = step
        self.max_samples_per_class = max_samples_per_class
        self.resample_each_epoch = resample_each_epoch
        
        # Tạo 2 mảng index: 1 cái step=1 (bảo toàn), 1 cái có step (băm mỏng)
        all_indices_step1 = np.arange(0, len(self.X) - self.time_steps + 1, 1)
        window_labels_step1 = self.y[all_indices_step1 + self.time_steps - 1].numpy()
        
        all_indices_stepped = np.arange(0, len(self.X) - self.time_steps + 1, self.step)
        window_labels_stepped = self.y[all_indices_stepped + self.time_steps - 1].numpy()
        
        self.class_indices = {}
        for c in np.unique(window_labels_step1):
            if c in rare_classes:
                # Bảo toàn 100% dữ liệu cho class hiếm
                self.class_indices[c] = all_indices_step1[window_labels_step1 == c]
            else:
                # Băm mỏng theo step đối với các class đa số
                self.class_indices[c] = all_indices_stepped[window_labels_stepped == c]
            
            print(f"Class {c}: Có sẵn {len(self.class_indices[c])} cửa sổ trong Pool")
            
        self._resample()
    
    def _resample(self):
        self.valid_indices = []
        for c, c_indices in self.class_indices.items():
            count = len(c_indices)
            
            # NẾU LÀ TẬP TRAIN (có bật resample_each_epoch)
            if self.resample_each_epoch:
                if count > self.max_samples_per_class:
                    # Dư thì băm đi (Undersampling)
                    sampled = np.random.choice(c_indices, self.max_samples_per_class, replace=False)
                else:
                    # Thiếu thì nhân bản (Oversampling)
                    sampled = np.random.choice(c_indices, self.max_samples_per_class, replace=True)
                    
            # NẾU LÀ TẬP VAL/TEST (Giữ nguyên gốc 100%, không thêm không bớt)
            else:
                sampled = c_indices
                
            self.valid_indices.extend(sampled)
            
        np.random.shuffle(self.valid_indices)
        self.valid_indices = np.array(self.valid_indices)
    def on_epoch_start(self):
        if self.resample_each_epoch:
            self._resample()

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        actual_idx = self.valid_indices[idx]
        window_X = self.X[actual_idx : actual_idx + self.time_steps]
        label_y = self.y[actual_idx + self.time_steps - 1]
        return window_X, label_y

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
from sklearn.model_selection import train_test_split
MAX_SAMPLES = 20000 
TIME_STEPS = 10
BATCH_SIZE = 256


TRAIN_STEP_SIZE = 5 
TEST_STEP_SIZE = 1
NUM_CLASSES = len(train_df['label'].unique())

# tối đa mỗi class sẽ có 20000 của sổ trượt
print(f"Khởi tạo tập Train (có Undersampling, set Train Step = {TRAIN_STEP_SIZE})...")
train_dataset = DynamicUndersampledSlidingWindowDataset(train_df, TIME_STEPS, max_samples_per_class=MAX_SAMPLES, step=TRAIN_STEP_SIZE, resample_each_epoch=True, rare_classes=[0, 2, 3, 4, 5, 8, 10])

print(f"Khởi tạo tập Val và Test (Không Undersampling, giữ nguyên gốc, set Test Step = {TEST_STEP_SIZE})...")
# để max_samples_per_class cho tập val và test lớn để không xóa cửa sổ trượt nào
val_dataset = DynamicUndersampledSlidingWindowDataset(valid_df, TIME_STEPS, max_samples_per_class=10000000, step=1, resample_each_epoch=False)
test_dataset = DynamicUndersampledSlidingWindowDataset(test_df, TIME_STEPS, max_samples_per_class=10000000, step=1, resample_each_epoch=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=2048, shuffle=True)

Khởi tạo tập Train (có Undersampling, set Train Step = 5)...
Class 0: Có sẵn 64488 cửa sổ trong Pool
Class 1: Có sẵn 314644 cửa sổ trong Pool
Class 2: Có sẵn 8164 cửa sổ trong Pool
Class 3: Có sẵn 117813 cửa sổ trong Pool
Class 4: Có sẵn 61666 cửa sổ trong Pool
Class 5: Có sẵn 7956 cửa sổ trong Pool
Class 6: Có sẵn 7698 cửa sổ trong Pool
Class 7: Có sẵn 69830 cửa sổ trong Pool
Class 8: Có sẵn 54420 cửa sổ trong Pool
Class 9: Có sẵn 26987 cửa sổ trong Pool
Class 10: Có sẵn 60328 cửa sổ trong Pool
Khởi tạo tập Val và Test (Không Undersampling, giữ nguyên gốc, set Test Step = 1)...
Class 0: Có sẵn 14880 cửa sổ trong Pool
Class 1: Có sẵn 363049 cửa sổ trong Pool
Class 2: Có sẵn 1884 cửa sổ trong Pool
Class 3: Có sẵn 27186 cửa sổ trong Pool
Class 4: Có sẵn 14229 cửa sổ trong Pool
Class 5: Có sẵn 1836 cửa sổ trong Pool
Class 6: Có sẵn 8880 cửa sổ trong Pool
Class 7: Có sẵn 80572 cửa sổ trong Pool
Class 8: Có sẵn 12558 cửa sổ trong Pool
Class 9: Có sẵn 31131 cửa sổ trong Pool
Class 10: Có sẵn

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
NUM_FEATURES = train_dataset.X.shape[1]

# CƠ CHẾ ATTENTION THEO THỜI GIAN (Temporal Attention)
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super(Attention, self).__init__()
        self.attention = nn.Linear(hidden_dim, 1)

    def forward(self, lstm_outputs):
        scores = self.attention(lstm_outputs)
        weights = torch.softmax(scores, dim=1)
        context_vector = torch.sum(weights * lstm_outputs, dim=1)
        return context_vector, weights

# VŨ KHÍ MỚI: Squeeze-and-Excitation (SE Block) - Chống Drift Kênh Đặc Trưng
class SEBlock1D(nn.Module):
    def __init__(self, channels, reduction=8):
        super(SEBlock1D, self).__init__()
        # Global Average Pooling theo chiều thời gian
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        # Bộ tạo trọng số động
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _ = x.size()
        y = self.avg_pool(x).view(b, c) # "Ngửi" toàn bộ cửa sổ để đánh giá độ tin cậy của từng kênh
        y = self.fc(y).view(b, c, 1)    # Tính ra thang điểm 0-1 cho từng kênh
        return x * y.expand_as(x)       # Bóp nghẹt kênh bị nhiễm Drift, Khuếch đại kênh chuẩn

# KHỐI RESIDUAL KẾT HỢP SE-BLOCK
class ResidualBlock1D(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3):
        super(ResidualBlock1D, self).__init__()
        padding = kernel_size // 2
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, padding=padding)
        self.gn1 = nn.GroupNorm(num_groups=8, num_channels=out_channels)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, padding=padding)
        self.gn2 = nn.GroupNorm(num_groups=8, num_channels=out_channels)
        self.dropout = nn.Dropout1d(0.2)
        
        # Nhúng cơ chế SE (Channel Attention)
        self.se = SEBlock1D(out_channels)
        
        # Đường tắt (Shortcut Connection)
        self.shortcut = nn.Sequential()
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1),
                nn.GroupNorm(num_groups=8, num_channels=out_channels)
            )
            
    def forward(self, x):
        residual = self.shortcut(x)
        out = self.conv1(x)
        out = self.gn1(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.conv2(out)
        out = self.gn2(out)
        
        # CHỐT CHẶN SE-BLOCK: Lọc bỏ tín hiệu rác do Concept Drift gây ra trên các biến (Features) mỏng manh
        out = self.se(out)
        
        out += residual  
        out = self.relu(out)
        return out

import torch
import torch.nn as nn
import torch.nn.functional as F

import torch
import torch.nn as nn
import torch.nn.functional as F

import torch
import torch.nn as nn
import torch.nn.functional as F

# [GIỮ NGUYÊN CODE CÁC LỚP: Attention, SEBlock1D, ResidualBlock1D]

# ---------------------------------------------------------
# 1. KHỐI PROCEED: ĐO LƯỜNG TRÔI DẠT (Kế thừa tư tưởng bài báo)
# ---------------------------------------------------------
class ConceptDriftEstimator(nn.Module):
    def __init__(self, feature_dim, momentum=0.99):
        super(ConceptDriftEstimator, self).__init__()
        self.momentum = momentum
        self.register_buffer('reference_mean', torch.zeros(feature_dim))

    def forward(self, current_features, is_training=False, clean_features=None):
        # features shape: (Batch, SeqLen, 128)
        
        # === SỬA ĐỔI LỚN TẠI ĐÂY ===
        # TÍNH MEAN THEO TỪNG CỬA SỔ (Instance-level) thay vì cả Batch
        current_mean = current_features.mean(dim=1) # Shape: (Batch, 128)
        
        if is_training and clean_features is not None:
            # Trí nhớ gốc vẫn phải được cập nhật bằng TRUNG BÌNH CỦA CẢ BATCH SẠCH
            batch_clean_mean = clean_features.mean(dim=(0, 1))
            self.reference_mean = self.momentum * self.reference_mean + (1 - self.momentum) * batch_clean_mean.detach()
        
        # Phép trừ Broadcasting: (Batch, 128) - (128,) -> (Batch, 128)
        # Sinh ra một vector độ lệch RIÊNG cho từng cửa sổ trong batch
        drift_vector = current_mean - self.reference_mean
        return drift_vector

# ---------------------------------------------------------
# 2. KHỐI PROCEED: BỘ SINH THÍCH ỨNG (MLP BOTTLENECK)
# ---------------------------------------------------------
class AdaptationGenerator(nn.Module):
    def __init__(self, drift_dim, bottleneck_dim=32):
        super(AdaptationGenerator, self).__init__()
        # Kiến trúc Bottleneck r=32 chuẩn bài báo để tránh Overfitting
        self.net = nn.Sequential(
            nn.Linear(drift_dim, bottleneck_dim),
            nn.Sigmoid(), # Hàm kích hoạt Sigmoid theo bài báo
            nn.Linear(bottleneck_dim, drift_dim * 2) # *2 để lấy cả Scale (Gamma) và Shift (Beta)
        )

    def forward(self, drift_vector):
        params = self.net(drift_vector)
        gamma, beta = params.chunk(2, dim=-1)
        # Giữ Gamma xoay quanh mức 1.0 (không đổi)
        gamma = torch.sigmoid(gamma) * 2.0 
        return gamma, beta

import torch
import torch.nn as nn
import torch.nn.functional as F

# [GIỮ NGUYÊN CODE CÁC LỚP: Attention, SEBlock1D, ResidualBlock1D, ConceptDriftEstimator, AdaptationGenerator]

# ---------------------------------------------------------
# KIẾN TRÚC MỚI: CNN-BiLSTM-ATTENTION + PROCEED E2E (WITH LOGGING)
# ---------------------------------------------------------
class Proceed_CNN_BiLSTM_Attention(nn.Module):
    def __init__(self, num_features, num_classes, time_steps, hidden_size=128):
        super(Proceed_CNN_BiLSTM_Attention, self).__init__()
        
        self.res1 = ResidualBlock1D(num_features, 64)
        self.res2 = ResidualBlock1D(64, 128)
        self.pool = nn.MaxPool1d(kernel_size=2) 
        
        self.drift_estimator = ConceptDriftEstimator(feature_dim=128)
        self.adapt_generator = AdaptationGenerator(drift_dim=128, bottleneck_dim=32)
        
        self.bilstm = nn.LSTM(input_size=128, hidden_size=hidden_size, 
                              batch_first=True, bidirectional=True)
        self.layer_norm = nn.LayerNorm(hidden_size * 2)
        self.attention = Attention(hidden_size * 2)
        self.dropout = nn.Dropout(0.5)
        
        self.fc1 = nn.Linear(hidden_size * 2, 64)
        self.fc_ln = nn.LayerNorm(64)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(64, num_classes)
        
        # Biến lưu trữ Log để tracking
        self.proceed_logs = {}
        
    def forward(self, x):
        x = x.permute(0, 2, 1) 
        x = self.res1(x)
        x = self.res2(x)
        x = self.pool(x)
        x_clean = x.permute(0, 2, 1) 
        
        is_noisy = False # Cờ theo dõi trạng thái nhiễu
        
        # ========================================================
        # CHẠY CƠ CHẾ PROCEED
        # ========================================================
        if self.training:
            if torch.rand(1).item() < 0.5:
                noise_scale = torch.empty(1, 1, 128, device=x_clean.device).uniform_(0.7, 1.3)
                noise_shift = torch.empty(1, 1, 128, device=x_clean.device).normal_(mean=0.0, std=0.5)
                x_target = (x_clean * noise_scale) + noise_shift
                is_noisy = True
            else:
                x_target = x_clean
                
            drift_vector = self.drift_estimator(x_target, is_training=True, clean_features=x_clean)
        else:
            x_target = x_clean
            drift_vector = self.drift_estimator(x_target, is_training=False)
            
        gamma, beta = self.adapt_generator(drift_vector)
        
        # --- BẮT ĐẦU GHI LOG ---
        # Tracking các chỉ số thống kê (không lưu gradient để tránh rò rỉ bộ nhớ)
        with torch.no_grad():
            self.proceed_logs = {
                'drift_norm': drift_vector.norm(p=2).item(), # Độ lớn của lực trôi dạt
                'gamma_mean': gamma.mean().item(),           # Trung bình hệ số Scale (Nên xoay quanh 1.0)
                'gamma_std':  gamma.std().item(),            # Độ lệch chuẩn Scale (Phải > 0, nếu = 0 là model đang lười)
                'beta_mean':  beta.mean().item(),            # Trung bình hệ số Shift (Nên xoay quanh 0.0)
                'beta_std':   beta.std().item(),             # Độ lệch chuẩn Shift
                'is_noisy':   is_noisy                       # Batch này có bị tiêm nhiễu ảo không?
            }
        # --- KẾT THÚC GHI LOG ---
        
        gamma = gamma.unsqueeze(1)
        beta = beta.unsqueeze(1)
        
        adapted_features = (x_target * gamma) + beta
        # ========================================================
        
        out, _ = self.bilstm(adapted_features)
        out = self.layer_norm(out)
        context_vector, attn_weights = self.attention(out)
        out = self.dropout(context_vector)
        out = self.fc1(out)
        out = self.fc_ln(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)
        
        return out



# KHỞI TẠO MÔ HÌNH MỚI
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Đang sử dụng thiết bị: {DEVICE}")

model = Proceed_CNN_BiLSTM_Attention(num_features=NUM_FEATURES, num_classes=NUM_CLASSES, time_steps=TIME_STEPS).to(DEVICE)
print(model)



Đang sử dụng thiết bị: cuda
Proceed_CNN_BiLSTM_Attention(
  (res1): ResidualBlock1D(
    (conv1): Conv1d(314, 64, kernel_size=(3,), stride=(1,), padding=(1,))
    (gn1): GroupNorm(8, 64, eps=1e-05, affine=True)
    (relu): ReLU()
    (conv2): Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding=(1,))
    (gn2): GroupNorm(8, 64, eps=1e-05, affine=True)
    (dropout): Dropout1d(p=0.2, inplace=False)
    (se): SEBlock1D(
      (avg_pool): AdaptiveAvgPool1d(output_size=1)
      (fc): Sequential(
        (0): Linear(in_features=64, out_features=8, bias=False)
        (1): ReLU(inplace=True)
        (2): Linear(in_features=8, out_features=64, bias=False)
        (3): Sigmoid()
      )
    )
    (shortcut): Sequential(
      (0): Conv1d(314, 64, kernel_size=(1,), stride=(1,))
      (1): GroupNorm(8, 64, eps=1e-05, affine=True)
    )
  )
  (res2): ResidualBlock1D(
    (conv1): Conv1d(64, 128, kernel_size=(3,), stride=(1,), padding=(1,))
    (gn1): GroupNorm(8, 128, eps=1e-05, affine=True)
   

In [6]:
from tqdm import tqdm
from sklearn.metrics import classification_report, f1_score
from tqdm import tqdm
from sklearn.metrics import classification_report, f1_score

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Trả LR về 0.0004 và Weight Decay về mức NHẸ 1e-3
optimizer = optim.Adam(model.parameters(), lr=0.0004, weight_decay=1e-3)

# Trả factor về 0.5 và patience về 3
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

EPOCHS = 30
best_val_f1 = 0.0 
patience_counter = 0
EARLY_STOP_PATIENCE = 12 # BẮT BUỘC LÀ 12 ĐỂ MODEL CÓ THỜI GIAN BỨT PHÁ



for epoch in range(EPOCHS):
    train_dataset.on_epoch_start()  
    model.train()
    train_loss = 0.0
    total_train = 0
    
    # Biến cộng dồn log PROCEED cho tập Train
    epoch_train_drift = 0.0
    epoch_train_gamma_std = 0.0
    
    all_train_preds = []
    all_train_targets = []
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    for inputs, labels in loop:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        
        # Áp dụng Gradient Clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        
        all_train_preds.extend(predicted.cpu().numpy())
        all_train_targets.extend(labels.cpu().numpy())
        
        # Bắt log PROCEED và cộng dồn
        logs = model.proceed_logs
        epoch_train_drift += logs['drift_norm']
        epoch_train_gamma_std += logs['gamma_std']
        
        loop.set_postfix(
            loss=loss.item(), 
            drift=f"{logs['drift_norm']:.3f}", 
            g_std=f"{logs['gamma_std']:.3f}",
            noisy=logs.get('is_noisy', False)
        )

    avg_train_loss = train_loss / total_train
    train_macro_f1 = f1_score(all_train_targets, all_train_preds, average='macro')
    avg_train_drift = epoch_train_drift / len(train_loader)
    avg_train_gamma_std = epoch_train_gamma_std / len(train_loader)
    
    
    model.eval()
    val_loss = 0.0
    total_val = 0
    
    # Biến cộng dồn log PROCEED cho tập Valid
    epoch_val_drift = 0.0
    epoch_val_gamma_std = 0.0
    
    all_val_preds = []
    all_val_targets = []
    
    val_loop = tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Validation]", leave=False)
    with torch.no_grad():
        for inputs, labels in val_loop:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            
            total_val += labels.size(0)
            
            all_val_preds.extend(predicted.cpu().numpy())
            all_val_targets.extend(labels.cpu().numpy())
            
            # Bắt log PROCEED và cộng dồn trên Valid
            logs = model.proceed_logs
            epoch_val_drift += logs['drift_norm']
            epoch_val_gamma_std += logs['gamma_std']
            
            # In log lên thanh tqdm của Valid (noisy luôn là False)
            val_loop.set_postfix(
                loss=loss.item(), 
                drift=f"{logs['drift_norm']:.3f}", 
                g_std=f"{logs['gamma_std']:.3f}"
            )

    avg_val_loss = val_loss / total_val
    val_macro_f1 = f1_score(all_val_targets, all_val_preds, average='macro')
    avg_val_drift = epoch_val_drift / len(val_loader)
    avg_val_gamma_std = epoch_val_gamma_std / len(val_loader)
    
    # ==========================================
    # IN BÁO CÁO CUỐI EPOCH
    # ==========================================
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f}, Train Macro F1: {train_macro_f1:.4f} | Val Loss: {avg_val_loss:.4f}, Val Macro F1: {val_macro_f1:.4f}")
    print(f"  └── [PROCEED] Train Drift: {avg_train_drift:.3f} (g_std: {avg_train_gamma_std:.3f}) | Val Drift: {avg_val_drift:.3f} (g_std: {avg_val_gamma_std:.3f})")
    
    scheduler.step(avg_val_loss)
    
    if val_macro_f1 > best_val_f1:
        print(f"  🌟 Chúc mừng! Val F1 tăng từ {best_val_f1:.4f} lên {val_macro_f1:.4f}. Đang lưu model...")
        best_val_f1 = val_macro_f1
        patience_counter = 0
        torch.save(model.state_dict(), "model_final/best_cnn_bilstm_1.pth")
    else:
        patience_counter += 1
        print(f"  ⚠️ Không cải thiện F1. Patience: {patience_counter}/{EARLY_STOP_PATIENCE}")
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\n[Early Stopping] Model đã bão hòa F1 sau {EARLY_STOP_PATIENCE} epochs.")
            break

print("\n--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST ---")
model.load_state_dict(torch.load("model_final/best_cnn_bilstm_1.pth", weights_only=True))
model.eval()

all_test_preds = []
all_test_targets = []

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="[Final Test]", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        all_test_preds.extend(predicted.cpu().numpy())
        all_test_targets.extend(labels.cpu().numpy())

print(classification_report(all_test_targets, all_test_preds))

Epoch [1/30] - Train Loss: 0.9116, Train Macro F1: 0.8625 | Val Loss: 0.6261, Val Macro F1: 0.8761
  └── [PROCEED] Train Drift: 148.536 (g_std: 0.102) | Val Drift: 127.226 (g_std: 0.061)
  🌟 Chúc mừng! Val F1 tăng từ 0.0000 lên 0.8761. Đang lưu model...


Epoch [2/30] - Train Loss: 0.7272, Train Macro F1: 0.9416 | Val Loss: 0.6081, Val Macro F1: 0.8989
  └── [PROCEED] Train Drift: 125.716 (g_std: 0.047) | Val Drift: 95.746 (g_std: 0.041)
  🌟 Chúc mừng! Val F1 tăng từ 0.8761 lên 0.8989. Đang lưu model...


Epoch [3/30] - Train Loss: 0.7016, Train Macro F1: 0.9560 | Val Loss: 0.5947, Val Macro F1: 0.9171
  └── [PROCEED] Train Drift: 102.564 (g_std: 0.043) | Val Drift: 73.760 (g_std: 0.054)
  🌟 Chúc mừng! Val F1 tăng từ 0.8989 lên 0.9171. Đang lưu model...


Epoch [4/30] - Train Loss: 0.7006, Train Macro F1: 0.9581 | Val Loss: 0.6085, Val Macro F1: 0.8976
  └── [PROCEED] Train Drift: 92.407 (g_std: 0.060) | Val Drift: 66.017 (g_std: 0.066)
  ⚠️ Không cải thiện F1. Patience: 1/12


Epoch [5/30] - Train Loss: 0.7002, Train Macro F1: 0.9596 | Val Loss: 0.6010, Val Macro F1: 0.9041
  └── [PROCEED] Train Drift: 90.647 (g_std: 0.071) | Val Drift: 66.572 (g_std: 0.071)
  ⚠️ Không cải thiện F1. Patience: 2/12


...

Epoch [27/30] - Train Loss: 0.6184, Train Macro F1: 0.9921 | Val Loss: 0.5820, Val Macro F1: 0.9391
  └── [PROCEED] Train Drift: 87.993 (g_std: 0.060) | Val Drift: 66.367 (g_std: 0.060)
  🌟 Chúc mừng! Val F1 tăng từ 0.9384 lên 0.9391. Đang lưu model...


Epoch [28/30] - Train Loss: 0.6171, Train Macro F1: 0.9920 | Val Loss: 0.5881, Val Macro F1: 0.9330
  └── [PROCEED] Train Drift: 86.801 (g_std: 0.060) | Val Drift: 66.165 (g_std: 0.060)
  ⚠️ Không cải thiện F1. Patience: 1/12


Epoch [29/30] - Train Loss: 0.6164, Train Macro F1: 0.9925 | Val Loss: 0.5778, Val Macro F1: 0.9400
  └── [PROCEED] Train Drift: 86.336 (g_std: 0.060) | Val Drift: 65.084 (g_std: 0.059)
  🌟 Chúc mừng! Val F1 tăng từ 0.9391 lên 0.9400. Đang lưu model...


Epoch [30/30] - Train Loss: 0.6168, Train Macro F1: 0.9923 | Val Loss: 0.5861, Val Macro F1: 0.9322
  └── [PROCEED] Train Drift: 86.518 (g_std: 0.060) | Val Drift: 65.063 (g_std: 0.060)
  ⚠️ Không cải thiện F1. Patience: 1/12

--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST ---


              precision    recall  f1-score   support

           0       0.90      0.82      0.86     19846
           1       1.00      1.00      1.00    484077
           2       0.37      0.97      0.53      2515
           3       1.00      0.83      0.91     36253
           4       0.72      0.86      0.78     18979
           5       0.99      1.00      0.99      2451
           6       0.66      0.73      0.69     11847
           7       1.00      1.00      1.00    107436
           8       0.70      0.99      0.82     16746
           9       1.00      0.67      0.80     41514
          10       0.78      0.99      0.87     18567

    accuracy                           0.96    760231
   macro avg       0.83      0.90      0.84    760231
weighted avg       0.97      0.96      0.96    760231



In [7]:
print("\n--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST ---")
model.load_state_dict(torch.load("model_final/best_cnn_bilstm_1.pth", weights_only=True))
model.eval()

all_test_preds = []
all_test_targets = []

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="[Final Test]", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        all_test_preds.extend(predicted.cpu().numpy())
        all_test_targets.extend(labels.cpu().numpy())

print(classification_report(all_test_targets, all_test_preds))


--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST ---


              precision    recall  f1-score   support

           0       0.90      0.82      0.86     19846
           1       1.00      1.00      1.00    484077
           2       0.37      0.97      0.53      2515
           3       1.00      0.83      0.91     36253
           4       0.72      0.86      0.78     18979
           5       0.99      1.00      0.99      2451
           6       0.66      0.73      0.69     11847
           7       1.00      1.00      1.00    107436
           8       0.70      0.99      0.82     16746
           9       1.00      0.67      0.80     41514
          10       0.78      0.99      0.87     18567

    accuracy                           0.96    760231
   macro avg       0.83      0.90      0.84    760231
weighted avg       0.97      0.96      0.96    760231



In [8]:
from tqdm import tqdm
from sklearn.metrics import classification_report, f1_score
from tqdm import tqdm
from sklearn.metrics import classification_report, f1_score
model = Proceed_CNN_BiLSTM_Attention(num_features=NUM_FEATURES, num_classes=NUM_CLASSES, time_steps=TIME_STEPS).to(DEVICE)
# Dùng Cross Entropy với Label Smoothing = 0.1 để chống over-confidence thay vì Focal Loss 
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Dùng Adam với weight decay nhỉnh hơn xíu (1e-3)
optimizer = optim.Adam(model.parameters(), lr=0.0004, weight_decay=1e-3)

# Trả factor về 0.5 và patience về 3
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

EPOCHS = 30
best_val_f1 = 0.0 
patience_counter = 0
EARLY_STOP_PATIENCE = 12 # BẮT BUỘC LÀ 12 ĐỂ MODEL CÓ THỜI GIAN BỨT PHÁ



for epoch in range(EPOCHS):
    train_dataset.on_epoch_start()  
    model.train()
    train_loss = 0.0
    total_train = 0
    
    # Biến cộng dồn log PROCEED cho tập Train
    epoch_train_drift = 0.0
    epoch_train_gamma_std = 0.0
    
    all_train_preds = []
    all_train_targets = []
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    for inputs, labels in loop:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        
        # Áp dụng Gradient Clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        
        all_train_preds.extend(predicted.cpu().numpy())
        all_train_targets.extend(labels.cpu().numpy())
        
        # Bắt log PROCEED và cộng dồn
        logs = model.proceed_logs
        epoch_train_drift += logs['drift_norm']
        epoch_train_gamma_std += logs['gamma_std']
        
        loop.set_postfix(
            loss=loss.item(), 
            drift=f"{logs['drift_norm']:.3f}", 
            g_std=f"{logs['gamma_std']:.3f}",
            noisy=logs.get('is_noisy', False)
        )

    avg_train_loss = train_loss / total_train
    train_macro_f1 = f1_score(all_train_targets, all_train_preds, average='macro')
    avg_train_drift = epoch_train_drift / len(train_loader)
    avg_train_gamma_std = epoch_train_gamma_std / len(train_loader)
    
    
    model.eval()
    val_loss = 0.0
    total_val = 0
    
    # Biến cộng dồn log PROCEED cho tập Valid
    epoch_val_drift = 0.0
    epoch_val_gamma_std = 0.0
    
    all_val_preds = []
    all_val_targets = []
    
    val_loop = tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Validation]", leave=False)
    with torch.no_grad():
        for inputs, labels in val_loop:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            
            total_val += labels.size(0)
            
            all_val_preds.extend(predicted.cpu().numpy())
            all_val_targets.extend(labels.cpu().numpy())
            
            # Bắt log PROCEED và cộng dồn trên Valid
            logs = model.proceed_logs
            epoch_val_drift += logs['drift_norm']
            epoch_val_gamma_std += logs['gamma_std']
            
            # In log lên thanh tqdm của Valid (noisy luôn là False)
            val_loop.set_postfix(
                loss=loss.item(), 
                drift=f"{logs['drift_norm']:.3f}", 
                g_std=f"{logs['gamma_std']:.3f}"
            )

    avg_val_loss = val_loss / total_val
    val_macro_f1 = f1_score(all_val_targets, all_val_preds, average='macro')
    avg_val_drift = epoch_val_drift / len(val_loader)
    avg_val_gamma_std = epoch_val_gamma_std / len(val_loader)
    
    # ==========================================
    # IN BÁO CÁO CUỐI EPOCH
    # ==========================================
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f}, Train Macro F1: {train_macro_f1:.4f} | Val Loss: {avg_val_loss:.4f}, Val Macro F1: {val_macro_f1:.4f}")
    print(f"  └── [PROCEED] Train Drift: {avg_train_drift:.3f} (g_std: {avg_train_gamma_std:.3f}) | Val Drift: {avg_val_drift:.3f} (g_std: {avg_val_gamma_std:.3f})")
    
    scheduler.step(avg_val_loss)
    
    if val_macro_f1 > best_val_f1:
        print(f"  🌟 Chúc mừng! Val F1 tăng từ {best_val_f1:.4f} lên {val_macro_f1:.4f}. Đang lưu model...")
        best_val_f1 = val_macro_f1
        patience_counter = 0
        torch.save(model.state_dict(), "model_final/best_cnn_bilstm_2.pth")
    else:
        patience_counter += 1
        print(f"  ⚠️ Không cải thiện F1. Patience: {patience_counter}/{EARLY_STOP_PATIENCE}")
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\n[Early Stopping] Model đã bão hòa F1 sau {EARLY_STOP_PATIENCE} epochs.")
            break

print("\n--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST ---")
model.load_state_dict(torch.load("model_final/best_cnn_bilstm_2.pth", weights_only=True))
model.eval()

all_test_preds = []
all_test_targets = []

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="[Final Test]", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        all_test_preds.extend(predicted.cpu().numpy())
        all_test_targets.extend(labels.cpu().numpy())

print(classification_report(all_test_targets, all_test_preds))

Epoch [1/30] - Train Loss: 0.9147, Train Macro F1: 0.8623 | Val Loss: 0.6232, Val Macro F1: 0.8788
  └── [PROCEED] Train Drift: 149.030 (g_std: 0.110) | Val Drift: 124.470 (g_std: 0.070)
  🌟 Chúc mừng! Val F1 tăng từ 0.0000 lên 0.8788. Đang lưu model...


Epoch [2/30] - Train Loss: 0.7107, Train Macro F1: 0.9515 | Val Loss: 0.6216, Val Macro F1: 0.8881
  └── [PROCEED] Train Drift: 123.365 (g_std: 0.053) | Val Drift: 96.084 (g_std: 0.042)
  🌟 Chúc mừng! Val F1 tăng từ 0.8788 lên 0.8881. Đang lưu model...


Epoch [3/30] - Train Loss: 0.6968, Train Macro F1: 0.9608 | Val Loss: 0.6032, Val Macro F1: 0.9041
  └── [PROCEED] Train Drift: 102.685 (g_std: 0.042) | Val Drift: 77.361 (g_std: 0.047)
  🌟 Chúc mừng! Val F1 tăng từ 0.8881 lên 0.9041. Đang lưu model...


Epoch [4/30] - Train Loss: 0.7033, Train Macro F1: 0.9608 | Val Loss: 0.6055, Val Macro F1: 0.9018
  └── [PROCEED] Train Drift: 91.023 (g_std: 0.058) | Val Drift: 69.327 (g_std: 0.062)
  ⚠️ Không cải thiện F1. Patience: 1/12


Epoch [5/30] - Train Loss: 0.7049, Train Macro F1: 0.9605 | Val Loss: 0.6107, Val Macro F1: 0.9042
  └── [PROCEED] Train Drift: 88.742 (g_std: 0.070) | Val Drift: 69.243 (g_std: 0.073)
  🌟 Chúc mừng! Val F1 tăng từ 0.9041 lên 0.9042. Đang lưu model...


...

Epoch [27/30] - Train Loss: 0.6079, Train Macro F1: 0.9956 | Val Loss: 0.5726, Val Macro F1: 0.9472
  └── [PROCEED] Train Drift: 85.908 (g_std: 0.051) | Val Drift: 62.904 (g_std: 0.050)
  🌟 Chúc mừng! Val F1 tăng từ 0.9457 lên 0.9472. Đang lưu model...


Epoch [28/30] - Train Loss: 0.6068, Train Macro F1: 0.9955 | Val Loss: 0.5754, Val Macro F1: 0.9445
  └── [PROCEED] Train Drift: 85.389 (g_std: 0.050) | Val Drift: 63.428 (g_std: 0.050)
  ⚠️ Không cải thiện F1. Patience: 1/12


Epoch [29/30] - Train Loss: 0.6050, Train Macro F1: 0.9961 | Val Loss: 0.5732, Val Macro F1: 0.9418
  └── [PROCEED] Train Drift: 87.517 (g_std: 0.050) | Val Drift: 62.952 (g_std: 0.050)
  ⚠️ Không cải thiện F1. Patience: 2/12


Epoch [30/30] - Train Loss: 0.6037, Train Macro F1: 0.9963 | Val Loss: 0.5711, Val Macro F1: 0.9467
  └── [PROCEED] Train Drift: 84.972 (g_std: 0.050) | Val Drift: 63.323 (g_std: 0.050)
  ⚠️ Không cải thiện F1. Patience: 3/12

--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST ---


              precision    recall  f1-score   support

           0       0.91      0.97      0.94     19846
           1       1.00      1.00      1.00    484077
           2       0.43      0.97      0.59      2515
           3       1.00      0.87      0.93     36253
           4       0.57      0.87      0.69     18979
           5       0.99      1.00      0.99      2451
           6       0.82      0.71      0.76     11847
           7       1.00      1.00      1.00    107436
           8       0.83      0.99      0.90     16746
           9       1.00      0.68      0.81     41514
          10       0.90      0.99      0.94     18567

    accuracy                           0.97    760231
   macro avg       0.86      0.91      0.87    760231
weighted avg       0.98      0.97      0.97    760231



In [9]:
from tqdm import tqdm
from sklearn.metrics import classification_report, f1_score
from tqdm import tqdm
from sklearn.metrics import classification_report, f1_score
model = Proceed_CNN_BiLSTM_Attention(num_features=NUM_FEATURES, num_classes=NUM_CLASSES, time_steps=TIME_STEPS).to(DEVICE)
# Dùng Cross Entropy với Label Smoothing = 0.1 để chống over-confidence thay vì Focal Loss 
optimizer = optim.Adam(model.parameters(), lr=0.0004, weight_decay=1e-3)

# Trả factor về 0.5 và patience về 3
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

EPOCHS = 30
best_val_f1 = 0.0
patience_counter = 0
EARLY_STOP_PATIENCE = 12

for epoch in range(EPOCHS):
    train_dataset.on_epoch_start()  
    model.train()
    train_loss = 0.0
    total_train = 0
    
    # Biến cộng dồn log PROCEED cho tập Train
    epoch_train_drift = 0.0
    epoch_train_gamma_std = 0.0
    
    all_train_preds = []
    all_train_targets = []
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    for inputs, labels in loop:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        
        # Áp dụng Gradient Clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        
        all_train_preds.extend(predicted.cpu().numpy())
        all_train_targets.extend(labels.cpu().numpy())
        
        # Bắt log PROCEED và cộng dồn
        logs = model.proceed_logs
        epoch_train_drift += logs['drift_norm']
        epoch_train_gamma_std += logs['gamma_std']
        
        loop.set_postfix(
            loss=loss.item(), 
            drift=f"{logs['drift_norm']:.3f}", 
            g_std=f"{logs['gamma_std']:.3f}",
            noisy=logs.get('is_noisy', False)
        )

    avg_train_loss = train_loss / total_train
    train_macro_f1 = f1_score(all_train_targets, all_train_preds, average='macro')
    avg_train_drift = epoch_train_drift / len(train_loader)
    avg_train_gamma_std = epoch_train_gamma_std / len(train_loader)
    
    
    model.eval()
    val_loss = 0.0
    total_val = 0
    
    # Biến cộng dồn log PROCEED cho tập Valid
    epoch_val_drift = 0.0
    epoch_val_gamma_std = 0.0
    
    all_val_preds = []
    all_val_targets = []
    
    val_loop = tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Validation]", leave=False)
    with torch.no_grad():
        for inputs, labels in val_loop:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            
            total_val += labels.size(0)
            
            all_val_preds.extend(predicted.cpu().numpy())
            all_val_targets.extend(labels.cpu().numpy())
            
            # Bắt log PROCEED và cộng dồn trên Valid
            logs = model.proceed_logs
            epoch_val_drift += logs['drift_norm']
            epoch_val_gamma_std += logs['gamma_std']
            
            # In log lên thanh tqdm của Valid (noisy luôn là False)
            val_loop.set_postfix(
                loss=loss.item(), 
                drift=f"{logs['drift_norm']:.3f}", 
                g_std=f"{logs['gamma_std']:.3f}"
            )

    avg_val_loss = val_loss / total_val
    val_macro_f1 = f1_score(all_val_targets, all_val_preds, average='macro')
    avg_val_drift = epoch_val_drift / len(val_loader)
    avg_val_gamma_std = epoch_val_gamma_std / len(val_loader)
    
    # ==========================================
    # IN BÁO CÁO CUỐI EPOCH
    # ==========================================
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f}, Train Macro F1: {train_macro_f1:.4f} | Val Loss: {avg_val_loss:.4f}, Val Macro F1: {val_macro_f1:.4f}")
    print(f"  └── [PROCEED] Train Drift: {avg_train_drift:.3f} (g_std: {avg_train_gamma_std:.3f}) | Val Drift: {avg_val_drift:.3f} (g_std: {avg_val_gamma_std:.3f})")
    
    scheduler.step(avg_val_loss)
    
    if val_macro_f1 > best_val_f1:
        print(f"  🌟 Chúc mừng! Val F1 tăng từ {best_val_f1:.4f} lên {val_macro_f1:.4f}. Đang lưu model...")
        best_val_f1 = val_macro_f1
        patience_counter = 0
        torch.save(model.state_dict(), "model_final/best_cnn_bilstm_3.pth")
    else:
        patience_counter += 1
        print(f"  ⚠️ Không cải thiện F1. Patience: {patience_counter}/{EARLY_STOP_PATIENCE}")
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\n[Early Stopping] Model đã bão hòa F1 sau {EARLY_STOP_PATIENCE} epochs.")
            break

print("\n--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST ---")
model.load_state_dict(torch.load("model_final/best_cnn_bilstm_3.pth", weights_only=True))
model.eval()

all_test_preds = []
all_test_targets = []

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="[Final Test]", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        all_test_preds.extend(predicted.cpu().numpy())
        all_test_targets.extend(labels.cpu().numpy())

print(classification_report(all_test_targets, all_test_preds))

Epoch [1/30] - Train Loss: 0.9230, Train Macro F1: 0.8583 | Val Loss: 0.6410, Val Macro F1: 0.8531
  └── [PROCEED] Train Drift: 147.487 (g_std: 0.102) | Val Drift: 125.848 (g_std: 0.063)
  🌟 Chúc mừng! Val F1 tăng từ 0.0000 lên 0.8531. Đang lưu model...


Epoch [2/30] - Train Loss: 0.7222, Train Macro F1: 0.9521 | Val Loss: 0.6156, Val Macro F1: 0.8989
  └── [PROCEED] Train Drift: 124.621 (g_std: 0.048) | Val Drift: 97.714 (g_std: 0.039)
  🌟 Chúc mừng! Val F1 tăng từ 0.8531 lên 0.8989. Đang lưu model...


Epoch [3/30] - Train Loss: 0.7167, Train Macro F1: 0.9554 | Val Loss: 0.6066, Val Macro F1: 0.9068
  └── [PROCEED] Train Drift: 103.108 (g_std: 0.041) | Val Drift: 75.988 (g_std: 0.044)
  🌟 Chúc mừng! Val F1 tăng từ 0.8989 lên 0.9068. Đang lưu model...


Epoch [4/30] - Train Loss: 0.7126, Train Macro F1: 0.9594 | Val Loss: 0.6182, Val Macro F1: 0.9130
  └── [PROCEED] Train Drift: 89.574 (g_std: 0.055) | Val Drift: 63.041 (g_std: 0.060)
  🌟 Chúc mừng! Val F1 tăng từ 0.9068 lên 0.9130. Đang lưu model...


Epoch [5/30] - Train Loss: 0.7098, Train Macro F1: 0.9609 | Val Loss: 0.6030, Val Macro F1: 0.9242
  └── [PROCEED] Train Drift: 87.249 (g_std: 0.065) | Val Drift: 69.300 (g_std: 0.070)
  🌟 Chúc mừng! Val F1 tăng từ 0.9130 lên 0.9242. Đang lưu model...


...

Epoch [14/30] - Train Loss: 0.6831, Train Macro F1: 0.9686 | Val Loss: 0.5921, Val Macro F1: 0.9167
  └── [PROCEED] Train Drift: 88.179 (g_std: 0.086) | Val Drift: 69.394 (g_std: 0.089)
  ⚠️ Không cải thiện F1. Patience: 2/5


Epoch [15/30] - Train Loss: 0.6849, Train Macro F1: 0.9679 | Val Loss: 0.5891, Val Macro F1: 0.9031
  └── [PROCEED] Train Drift: 88.851 (g_std: 0.086) | Val Drift: 68.046 (g_std: 0.083)
  ⚠️ Không cải thiện F1. Patience: 3/5


Epoch [16/30] - Train Loss: 0.6827, Train Macro F1: 0.9687 | Val Loss: 0.5762, Val Macro F1: 0.9315
  └── [PROCEED] Train Drift: 87.327 (g_std: 0.089) | Val Drift: 68.262 (g_std: 0.091)
  ⚠️ Không cải thiện F1. Patience: 4/5


Epoch [17/30] - Train Loss: 0.6591, Train Macro F1: 0.9768 | Val Loss: 0.5905, Val Macro F1: 0.9116
  └── [PROCEED] Train Drift: 89.354 (g_std: 0.086) | Val Drift: 69.886 (g_std: 0.080)
  ⚠️ Không cải thiện F1. Patience: 5/5

[Early Stopping] Model đã bão hòa F1 sau 5 epochs.

--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST ---


              precision    recall  f1-score   support

           0       0.77      0.77      0.77     19846
           1       1.00      1.00      1.00    484077
           2       0.25      1.00      0.39      2515
           3       1.00      0.78      0.88     36253
           4       0.61      0.77      0.68     18979
           5       1.00      0.99      1.00      2451
           6       0.73      0.73      0.73     11847
           7       1.00      0.99      1.00    107436
           8       0.72      0.95      0.82     16746
           9       1.00      0.74      0.85     41514
          10       0.92      0.99      0.95     18567

    accuracy                           0.96    760231
   macro avg       0.82      0.88      0.82    760231
weighted avg       0.97      0.96      0.96    760231



In [10]:
from tqdm import tqdm
from sklearn.metrics import classification_report, f1_score
from tqdm import tqdm
from sklearn.metrics import classification_report, f1_score
model = Proceed_CNN_BiLSTM_Attention(num_features=NUM_FEATURES, num_classes=NUM_CLASSES, time_steps=TIME_STEPS).to(DEVICE)
# Dùng Cross Entropy với Label Smoothing = 0.1 để chống over-confidence thay vì Focal Loss 
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = optim.Adam(model.parameters(), lr=0.0004, weight_decay=1e-3)

# Trả factor về 0.5 và patience về 3
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

EPOCHS = 30
best_val_f1 = 0.0
patience_counter = 0
EARLY_STOP_PATIENCE = 12

for epoch in range(EPOCHS):
    train_dataset.on_epoch_start()  
    model.train()
    train_loss = 0.0
    total_train = 0
    
    # Biến cộng dồn log PROCEED cho tập Train
    epoch_train_drift = 0.0
    epoch_train_gamma_std = 0.0
    
    all_train_preds = []
    all_train_targets = []
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    for inputs, labels in loop:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        
        # Áp dụng Gradient Clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        
        all_train_preds.extend(predicted.cpu().numpy())
        all_train_targets.extend(labels.cpu().numpy())
        
        # Bắt log PROCEED và cộng dồn
        logs = model.proceed_logs
        epoch_train_drift += logs['drift_norm']
        epoch_train_gamma_std += logs['gamma_std']
        
        loop.set_postfix(
            loss=loss.item(), 
            drift=f"{logs['drift_norm']:.3f}", 
            g_std=f"{logs['gamma_std']:.3f}",
            noisy=logs.get('is_noisy', False)
        )

    avg_train_loss = train_loss / total_train
    train_macro_f1 = f1_score(all_train_targets, all_train_preds, average='macro')
    avg_train_drift = epoch_train_drift / len(train_loader)
    avg_train_gamma_std = epoch_train_gamma_std / len(train_loader)
    
    
    model.eval()
    val_loss = 0.0
    total_val = 0
    
    # Biến cộng dồn log PROCEED cho tập Valid
    epoch_val_drift = 0.0
    epoch_val_gamma_std = 0.0
    
    all_val_preds = []
    all_val_targets = []
    
    val_loop = tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Validation]", leave=False)
    with torch.no_grad():
        for inputs, labels in val_loop:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            
            total_val += labels.size(0)
            
            all_val_preds.extend(predicted.cpu().numpy())
            all_val_targets.extend(labels.cpu().numpy())
            
            # Bắt log PROCEED và cộng dồn trên Valid
            logs = model.proceed_logs
            epoch_val_drift += logs['drift_norm']
            epoch_val_gamma_std += logs['gamma_std']
            
            # In log lên thanh tqdm của Valid (noisy luôn là False)
            val_loop.set_postfix(
                loss=loss.item(), 
                drift=f"{logs['drift_norm']:.3f}", 
                g_std=f"{logs['gamma_std']:.3f}"
            )

    avg_val_loss = val_loss / total_val
    val_macro_f1 = f1_score(all_val_targets, all_val_preds, average='macro')
    avg_val_drift = epoch_val_drift / len(val_loader)
    avg_val_gamma_std = epoch_val_gamma_std / len(val_loader)
    
    # ==========================================
    # IN BÁO CÁO CUỐI EPOCH
    # ==========================================
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f}, Train Macro F1: {train_macro_f1:.4f} | Val Loss: {avg_val_loss:.4f}, Val Macro F1: {val_macro_f1:.4f}")
    print(f"  └── [PROCEED] Train Drift: {avg_train_drift:.3f} (g_std: {avg_train_gamma_std:.3f}) | Val Drift: {avg_val_drift:.3f} (g_std: {avg_val_gamma_std:.3f})")
    
    scheduler.step(avg_val_loss)
    
    if val_macro_f1 > best_val_f1:
        print(f"  🌟 Chúc mừng! Val F1 tăng từ {best_val_f1:.4f} lên {val_macro_f1:.4f}. Đang lưu model...")
        best_val_f1 = val_macro_f1
        patience_counter = 0
        torch.save(model.state_dict(), "model_final/best_cnn_bilstm_4.pth")
    else:
        patience_counter += 1
        print(f"  ⚠️ Không cải thiện F1. Patience: {patience_counter}/{EARLY_STOP_PATIENCE}")
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\n[Early Stopping] Model đã bão hòa F1 sau {EARLY_STOP_PATIENCE} epochs.")
            break

print("\n--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST ---")
model.load_state_dict(torch.load("model_final/best_cnn_bilstm_4.pth", weights_only=True))
model.eval()

all_test_preds = []
all_test_targets = []

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="[Final Test]", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        all_test_preds.extend(predicted.cpu().numpy())
        all_test_targets.extend(labels.cpu().numpy())

print(classification_report(all_test_targets, all_test_preds))

Epoch [1/30] - Train Loss: 0.9044, Train Macro F1: 0.8701 | Val Loss: 0.6286, Val Macro F1: 0.8770
  └── [PROCEED] Train Drift: 145.305 (g_std: 0.104) | Val Drift: 127.760 (g_std: 0.063)
  🌟 Chúc mừng! Val F1 tăng từ 0.0000 lên 0.8770. Đang lưu model...


Epoch [2/30] - Train Loss: 0.7148, Train Macro F1: 0.9516 | Val Loss: 0.6008, Val Macro F1: 0.9142
  └── [PROCEED] Train Drift: 123.183 (g_std: 0.051) | Val Drift: 101.730 (g_std: 0.047)
  🌟 Chúc mừng! Val F1 tăng từ 0.8770 lên 0.9142. Đang lưu model...


Epoch [3/30] - Train Loss: 0.7057, Train Macro F1: 0.9570 | Val Loss: 0.5946, Val Macro F1: 0.9132
  └── [PROCEED] Train Drift: 102.945 (g_std: 0.048) | Val Drift: 76.341 (g_std: 0.051)
  ⚠️ Không cải thiện F1. Patience: 1/12


Epoch [4/30] - Train Loss: 0.7011, Train Macro F1: 0.9607 | Val Loss: 0.5976, Val Macro F1: 0.9144
  └── [PROCEED] Train Drift: 91.535 (g_std: 0.063) | Val Drift: 71.457 (g_std: 0.072)
  🌟 Chúc mừng! Val F1 tăng từ 0.9142 lên 0.9144. Đang lưu model...


Epoch [5/30] - Train Loss: 0.7004, Train Macro F1: 0.9605 | Val Loss: 0.6013, Val Macro F1: 0.9200
  └── [PROCEED] Train Drift: 90.333 (g_std: 0.070) | Val Drift: 70.829 (g_std: 0.071)
  🌟 Chúc mừng! Val F1 tăng từ 0.9144 lên 0.9200. Đang lưu model...


...

Epoch [27/30] - Train Loss: 0.6127, Train Macro F1: 0.9940 | Val Loss: 0.5762, Val Macro F1: 0.9385
  └── [PROCEED] Train Drift: 85.505 (g_std: 0.053) | Val Drift: 65.381 (g_std: 0.053)
  ⚠️ Không cải thiện F1. Patience: 5/12


Epoch [28/30] - Train Loss: 0.6126, Train Macro F1: 0.9944 | Val Loss: 0.5874, Val Macro F1: 0.9318
  └── [PROCEED] Train Drift: 85.187 (g_std: 0.053) | Val Drift: 65.228 (g_std: 0.053)
  ⚠️ Không cải thiện F1. Patience: 6/12


Epoch [29/30] - Train Loss: 0.6125, Train Macro F1: 0.9941 | Val Loss: 0.5823, Val Macro F1: 0.9340
  └── [PROCEED] Train Drift: 85.868 (g_std: 0.053) | Val Drift: 64.779 (g_std: 0.052)
  ⚠️ Không cải thiện F1. Patience: 7/12


Epoch [30/30] - Train Loss: 0.6097, Train Macro F1: 0.9953 | Val Loss: 0.5752, Val Macro F1: 0.9424
  └── [PROCEED] Train Drift: 87.496 (g_std: 0.052) | Val Drift: 65.188 (g_std: 0.052)
  🌟 Chúc mừng! Val F1 tăng từ 0.9404 lên 0.9424. Đang lưu model...

--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST ---


              precision    recall  f1-score   support

           0       0.91      0.82      0.86     19846
           1       1.00      1.00      1.00    484077
           2       0.33      0.97      0.50      2515
           3       0.99      0.86      0.92     36253
           4       0.68      0.87      0.77     18979
           5       0.93      1.00      0.96      2451
           6       0.69      0.72      0.71     11847
           7       1.00      1.00      1.00    107436
           8       0.71      0.99      0.83     16746
           9       1.00      0.66      0.80     41514
          10       0.86      0.99      0.92     18567

    accuracy                           0.96    760231
   macro avg       0.83      0.90      0.84    760231
weighted avg       0.97      0.96      0.96    760231



In [11]:
from tqdm import tqdm
from sklearn.metrics import classification_report, f1_score
from tqdm import tqdm
from sklearn.metrics import classification_report, f1_score
model = Proceed_CNN_BiLSTM_Attention(num_features=NUM_FEATURES, num_classes=NUM_CLASSES, time_steps=TIME_STEPS).to(DEVICE)
# Dùng Cross Entropy với Label Smoothing = 0.1 để chống over-confidence thay vì Focal Loss 
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Dùng Adam với weight decay nhỉnh hơn xíu (1e-3)
optimizer = optim.Adam(model.parameters(), lr=0.0004, weight_decay=1e-3)

# Trả factor về 0.5 và patience về 3
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

EPOCHS = 30
best_val_f1 = 0.0
patience_counter = 0
EARLY_STOP_PATIENCE = 12

for epoch in range(EPOCHS):
    train_dataset.on_epoch_start()  
    model.train()
    train_loss = 0.0
    total_train = 0
    
    # Biến cộng dồn log PROCEED cho tập Train
    epoch_train_drift = 0.0
    epoch_train_gamma_std = 0.0
    
    all_train_preds = []
    all_train_targets = []
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    for inputs, labels in loop:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        
        # Áp dụng Gradient Clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        
        all_train_preds.extend(predicted.cpu().numpy())
        all_train_targets.extend(labels.cpu().numpy())
        
        # Bắt log PROCEED và cộng dồn
        logs = model.proceed_logs
        epoch_train_drift += logs['drift_norm']
        epoch_train_gamma_std += logs['gamma_std']
        
        loop.set_postfix(
            loss=loss.item(), 
            drift=f"{logs['drift_norm']:.3f}", 
            g_std=f"{logs['gamma_std']:.3f}",
            noisy=logs.get('is_noisy', False)
        )

    avg_train_loss = train_loss / total_train
    train_macro_f1 = f1_score(all_train_targets, all_train_preds, average='macro')
    avg_train_drift = epoch_train_drift / len(train_loader)
    avg_train_gamma_std = epoch_train_gamma_std / len(train_loader)
    
    
    model.eval()
    val_loss = 0.0
    total_val = 0
    
    # Biến cộng dồn log PROCEED cho tập Valid
    epoch_val_drift = 0.0
    epoch_val_gamma_std = 0.0
    
    all_val_preds = []
    all_val_targets = []
    
    val_loop = tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Validation]", leave=False)
    with torch.no_grad():
        for inputs, labels in val_loop:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            
            total_val += labels.size(0)
            
            all_val_preds.extend(predicted.cpu().numpy())
            all_val_targets.extend(labels.cpu().numpy())
            
            # Bắt log PROCEED và cộng dồn trên Valid
            logs = model.proceed_logs
            epoch_val_drift += logs['drift_norm']
            epoch_val_gamma_std += logs['gamma_std']
            
            # In log lên thanh tqdm của Valid (noisy luôn là False)
            val_loop.set_postfix(
                loss=loss.item(), 
                drift=f"{logs['drift_norm']:.3f}", 
                g_std=f"{logs['gamma_std']:.3f}"
            )

    avg_val_loss = val_loss / total_val
    val_macro_f1 = f1_score(all_val_targets, all_val_preds, average='macro')
    avg_val_drift = epoch_val_drift / len(val_loader)
    avg_val_gamma_std = epoch_val_gamma_std / len(val_loader)
    
    # ==========================================
    # IN BÁO CÁO CUỐI EPOCH
    # ==========================================
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f}, Train Macro F1: {train_macro_f1:.4f} | Val Loss: {avg_val_loss:.4f}, Val Macro F1: {val_macro_f1:.4f}")
    print(f"  └── [PROCEED] Train Drift: {avg_train_drift:.3f} (g_std: {avg_train_gamma_std:.3f}) | Val Drift: {avg_val_drift:.3f} (g_std: {avg_val_gamma_std:.3f})")
    
    scheduler.step(avg_val_loss)
    
    if val_macro_f1 > best_val_f1:
        print(f"  🌟 Chúc mừng! Val F1 tăng từ {best_val_f1:.4f} lên {val_macro_f1:.4f}. Đang lưu model...")
        best_val_f1 = val_macro_f1
        patience_counter = 0
        torch.save(model.state_dict(), "model_final/best_cnn_bilstm_5.pth")
    else:
        patience_counter += 1
        print(f"  ⚠️ Không cải thiện F1. Patience: {patience_counter}/{EARLY_STOP_PATIENCE}")
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\n[Early Stopping] Model đã bão hòa F1 sau {EARLY_STOP_PATIENCE} epochs.")
            break

print("\n--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST ---")
model.load_state_dict(torch.load("model_final/best_cnn_bilstm_5.pth", weights_only=True))
model.eval()

all_test_preds = []
all_test_targets = []

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="[Final Test]", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        all_test_preds.extend(predicted.cpu().numpy())
        all_test_targets.extend(labels.cpu().numpy())

print(classification_report(all_test_targets, all_test_preds))

Epoch [1/30] - Train Loss: 0.9127, Train Macro F1: 0.8622 | Val Loss: 0.6053, Val Macro F1: 0.9084
  └── [PROCEED] Train Drift: 145.411 (g_std: 0.103) | Val Drift: 126.944 (g_std: 0.062)
  🌟 Chúc mừng! Val F1 tăng từ 0.0000 lên 0.9084. Đang lưu model...


Epoch [2/30] - Train Loss: 0.7144, Train Macro F1: 0.9515 | Val Loss: 0.5906, Val Macro F1: 0.9204
  └── [PROCEED] Train Drift: 123.193 (g_std: 0.048) | Val Drift: 97.764 (g_std: 0.040)
  🌟 Chúc mừng! Val F1 tăng từ 0.9084 lên 0.9204. Đang lưu model...


Epoch [3/30] - Train Loss: 0.7075, Train Macro F1: 0.9565 | Val Loss: 0.6034, Val Macro F1: 0.9059
  └── [PROCEED] Train Drift: 102.666 (g_std: 0.040) | Val Drift: 75.853 (g_std: 0.045)
  ⚠️ Không cải thiện F1. Patience: 1/12


Epoch [4/30] - Train Loss: 0.7143, Train Macro F1: 0.9565 | Val Loss: 0.5794, Val Macro F1: 0.9233
  └── [PROCEED] Train Drift: 91.603 (g_std: 0.057) | Val Drift: 69.882 (g_std: 0.061)
  🌟 Chúc mừng! Val F1 tăng từ 0.9204 lên 0.9233. Đang lưu model...


Epoch [5/30] - Train Loss: 0.7093, Train Macro F1: 0.9586 | Val Loss: 0.5949, Val Macro F1: 0.9185
  └── [PROCEED] Train Drift: 90.977 (g_std: 0.070) | Val Drift: 68.349 (g_std: 0.069)
  ⚠️ Không cải thiện F1. Patience: 1/12


...

Epoch [27/30] - Train Loss: 0.6299, Train Macro F1: 0.9864 | Val Loss: 0.5792, Val Macro F1: 0.9290
  └── [PROCEED] Train Drift: 87.839 (g_std: 0.065) | Val Drift: 68.050 (g_std: 0.065)
  ⚠️ Không cải thiện F1. Patience: 3/12


Epoch [28/30] - Train Loss: 0.6278, Train Macro F1: 0.9867 | Val Loss: 0.5785, Val Macro F1: 0.9297
  └── [PROCEED] Train Drift: 87.167 (g_std: 0.065) | Val Drift: 67.478 (g_std: 0.064)
  ⚠️ Không cải thiện F1. Patience: 4/12


Epoch [29/30] - Train Loss: 0.6277, Train Macro F1: 0.9869 | Val Loss: 0.5762, Val Macro F1: 0.9329
  └── [PROCEED] Train Drift: 88.956 (g_std: 0.064) | Val Drift: 67.887 (g_std: 0.064)
  ⚠️ Không cải thiện F1. Patience: 5/12


Epoch [30/30] - Train Loss: 0.6264, Train Macro F1: 0.9878 | Val Loss: 0.5789, Val Macro F1: 0.9306
  └── [PROCEED] Train Drift: 87.774 (g_std: 0.063) | Val Drift: 67.447 (g_std: 0.063)
  ⚠️ Không cải thiện F1. Patience: 6/12

--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST ---


              precision    recall  f1-score   support

           0       0.88      0.79      0.83     19846
           1       1.00      1.00      1.00    484077
           2       0.24      0.99      0.39      2515
           3       0.99      0.80      0.88     36253
           4       0.66      0.86      0.74     18979
           5       0.98      1.00      0.99      2451
           6       0.65      0.73      0.69     11847
           7       1.00      0.99      1.00    107436
           8       0.71      0.96      0.82     16746
           9       1.00      0.69      0.82     41514
          10       0.90      0.99      0.94     18567

    accuracy                           0.96    760231
   macro avg       0.82      0.89      0.83    760231
weighted avg       0.97      0.96      0.96    760231



In [12]:
from tqdm import tqdm
from sklearn.metrics import classification_report, f1_score
from tqdm import tqdm
from sklearn.metrics import classification_report, f1_score
model = Proceed_CNN_BiLSTM_Attention(num_features=NUM_FEATURES, num_classes=NUM_CLASSES, time_steps=TIME_STEPS).to(DEVICE)
# Dùng Cross Entropy với Label Smoothing = 0.1 để chống over-confidence thay vì Focal Loss 
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Dùng Adam với weight decay nhỉnh hơn xíu (1e-3)
optimizer = optim.Adam(model.parameters(), lr=0.0004, weight_decay=1e-3)

# Trả factor về 0.5 và patience về 3
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

EPOCHS = 30
best_val_f1 = 0.0
patience_counter = 0
EARLY_STOP_PATIENCE = 12

for epoch in range(EPOCHS):
    train_dataset.on_epoch_start()  
    model.train()
    train_loss = 0.0
    total_train = 0
    
    # Biến cộng dồn log PROCEED cho tập Train
    epoch_train_drift = 0.0
    epoch_train_gamma_std = 0.0
    
    all_train_preds = []
    all_train_targets = []
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    for inputs, labels in loop:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        
        # Áp dụng Gradient Clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        
        all_train_preds.extend(predicted.cpu().numpy())
        all_train_targets.extend(labels.cpu().numpy())
        
        # Bắt log PROCEED và cộng dồn
        logs = model.proceed_logs
        epoch_train_drift += logs['drift_norm']
        epoch_train_gamma_std += logs['gamma_std']
        
        loop.set_postfix(
            loss=loss.item(), 
            drift=f"{logs['drift_norm']:.3f}", 
            g_std=f"{logs['gamma_std']:.3f}",
            noisy=logs.get('is_noisy', False)
        )

    avg_train_loss = train_loss / total_train
    train_macro_f1 = f1_score(all_train_targets, all_train_preds, average='macro')
    avg_train_drift = epoch_train_drift / len(train_loader)
    avg_train_gamma_std = epoch_train_gamma_std / len(train_loader)
    
    
    model.eval()
    val_loss = 0.0
    total_val = 0
    
    # Biến cộng dồn log PROCEED cho tập Valid
    epoch_val_drift = 0.0
    epoch_val_gamma_std = 0.0
    
    all_val_preds = []
    all_val_targets = []
    
    val_loop = tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Validation]", leave=False)
    with torch.no_grad():
        for inputs, labels in val_loop:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            
            total_val += labels.size(0)
            
            all_val_preds.extend(predicted.cpu().numpy())
            all_val_targets.extend(labels.cpu().numpy())
            
            # Bắt log PROCEED và cộng dồn trên Valid
            logs = model.proceed_logs
            epoch_val_drift += logs['drift_norm']
            epoch_val_gamma_std += logs['gamma_std']
            
            # In log lên thanh tqdm của Valid (noisy luôn là False)
            val_loop.set_postfix(
                loss=loss.item(), 
                drift=f"{logs['drift_norm']:.3f}", 
                g_std=f"{logs['gamma_std']:.3f}"
            )

    avg_val_loss = val_loss / total_val
    val_macro_f1 = f1_score(all_val_targets, all_val_preds, average='macro')
    avg_val_drift = epoch_val_drift / len(val_loader)
    avg_val_gamma_std = epoch_val_gamma_std / len(val_loader)
    
    # ==========================================
    # IN BÁO CÁO CUỐI EPOCH
    # ==========================================
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f}, Train Macro F1: {train_macro_f1:.4f} | Val Loss: {avg_val_loss:.4f}, Val Macro F1: {val_macro_f1:.4f}")
    print(f"  └── [PROCEED] Train Drift: {avg_train_drift:.3f} (g_std: {avg_train_gamma_std:.3f}) | Val Drift: {avg_val_drift:.3f} (g_std: {avg_val_gamma_std:.3f})")
    
    scheduler.step(avg_val_loss)
    
    if val_macro_f1 > best_val_f1:
        print(f"  🌟 Chúc mừng! Val F1 tăng từ {best_val_f1:.4f} lên {val_macro_f1:.4f}. Đang lưu model...")
        best_val_f1 = val_macro_f1
        patience_counter = 0
        torch.save(model.state_dict(), "model_final/best_cnn_bilstm_6.pth")
    else:
        patience_counter += 1
        print(f"  ⚠️ Không cải thiện F1. Patience: {patience_counter}/{EARLY_STOP_PATIENCE}")
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\n[Early Stopping] Model đã bão hòa F1 sau {EARLY_STOP_PATIENCE} epochs.")
            break

print("\n--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST ---")
model.load_state_dict(torch.load("model_final/best_cnn_bilstm_6.pth", weights_only=True))
model.eval()

all_test_preds = []
all_test_targets = []

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="[Final Test]", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        all_test_preds.extend(predicted.cpu().numpy())
        all_test_targets.extend(labels.cpu().numpy())

print(classification_report(all_test_targets, all_test_preds))

Epoch [1/30] - Train Loss: 0.9132, Train Macro F1: 0.8624 | Val Loss: 0.6138, Val Macro F1: 0.8974
  └── [PROCEED] Train Drift: 144.782 (g_std: 0.093) | Val Drift: 122.293 (g_std: 0.064)
  🌟 Chúc mừng! Val F1 tăng từ 0.0000 lên 0.8974. Đang lưu model...


Epoch [2/30] - Train Loss: 0.7130, Train Macro F1: 0.9509 | Val Loss: 0.5983, Val Macro F1: 0.9136
  └── [PROCEED] Train Drift: 122.959 (g_std: 0.050) | Val Drift: 94.444 (g_std: 0.042)
  🌟 Chúc mừng! Val F1 tăng từ 0.8974 lên 0.9136. Đang lưu model...


Epoch [3/30] - Train Loss: 0.7000, Train Macro F1: 0.9590 | Val Loss: 0.6147, Val Macro F1: 0.8788
  └── [PROCEED] Train Drift: 101.256 (g_std: 0.043) | Val Drift: 74.615 (g_std: 0.047)
  ⚠️ Không cải thiện F1. Patience: 1/12


Epoch [4/30] - Train Loss: 0.7065, Train Macro F1: 0.9588 | Val Loss: 0.6035, Val Macro F1: 0.8932
  └── [PROCEED] Train Drift: 91.870 (g_std: 0.055) | Val Drift: 66.864 (g_std: 0.057)
  ⚠️ Không cải thiện F1. Patience: 2/12


Epoch [5/30] - Train Loss: 0.7064, Train Macro F1: 0.9596 | Val Loss: 0.6103, Val Macro F1: 0.9017
  └── [PROCEED] Train Drift: 91.467 (g_std: 0.069) | Val Drift: 68.343 (g_std: 0.070)
  ⚠️ Không cải thiện F1. Patience: 3/12


...

Epoch [19/30] - Train Loss: 0.6687, Train Macro F1: 0.9792 | Val Loss: 0.5803, Val Macro F1: 0.9399
  └── [PROCEED] Train Drift: 87.141 (g_std: 0.067) | Val Drift: 64.608 (g_std: 0.071)
  ⚠️ Không cải thiện F1. Patience: 9/12


Epoch [20/30] - Train Loss: 0.6713, Train Macro F1: 0.9787 | Val Loss: 0.5846, Val Macro F1: 0.9388
  └── [PROCEED] Train Drift: 86.622 (g_std: 0.068) | Val Drift: 64.426 (g_std: 0.065)
  ⚠️ Không cải thiện F1. Patience: 10/12


Epoch [21/30] - Train Loss: 0.6764, Train Macro F1: 0.9772 | Val Loss: 0.5932, Val Macro F1: 0.9294
  └── [PROCEED] Train Drift: 86.903 (g_std: 0.066) | Val Drift: 60.777 (g_std: 0.067)
  ⚠️ Không cải thiện F1. Patience: 11/12


Epoch [22/30] - Train Loss: 0.6378, Train Macro F1: 0.9876 | Val Loss: 0.5878, Val Macro F1: 0.9170
  └── [PROCEED] Train Drift: 84.532 (g_std: 0.067) | Val Drift: 63.934 (g_std: 0.066)
  ⚠️ Không cải thiện F1. Patience: 12/12

[Early Stopping] Model đã bão hòa F1 sau 12 epochs.

--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST ---


              precision    recall  f1-score   support

           0       0.94      0.78      0.85     19846
           1       1.00      1.00      1.00    484077
           2       0.41      1.00      0.59      2515
           3       1.00      0.89      0.94     36253
           4       0.51      0.84      0.63     18979
           5       0.99      1.00      0.99      2451
           6       0.87      0.69      0.77     11847
           7       1.00      0.99      1.00    107436
           8       0.67      0.98      0.80     16746
           9       1.00      0.67      0.80     41514
          10       0.96      0.99      0.97     18567

    accuracy                           0.96    760231
   macro avg       0.85      0.89      0.85    760231
weighted avg       0.97      0.96      0.96    760231



In [ ]:
# load lại model và test với tập test không shuffle để xem kết quả chi tiết hơn
model.load_state_dict(torch.load("model_final/best_cnn_bilstm_v2.pth", weights_only=True))
model.eval()
all_test_preds = []
all_test_targets = []
test_dataset = DynamicUndersampledSlidingWindowDataset(test_df, TIME_STEPS, max_samples_per_class=10000000, step=1, resample_each_epoch=False)
test_loader = DataLoader(test_dataset, batch_size=2048, shuffle=False)
with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="[Final Test - No Shuffle]", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        all_test_preds.extend(predicted.cpu().numpy())
        all_test_targets.extend(labels.cpu().numpy())
print(classification_report(all_test_targets, all_test_preds))  